In [1]:
import json
import requests
import numpy as np
import pandas as pd
import pandas as pd
import numpy as np
from datetime import datetime
from sqlalchemy import create_engine, text
import os
import json
import tempfile
import mlflow
import mlflow.catboost
from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import os
from mlflow import MlflowClient
import tempfile
from sklearn.model_selection import train_test_split

In [2]:
FEATURES = [

    "status",
    "city",
    "state",
    "zip_code",

    "bed",
    "bath",
    "acre_lot",
    "house_size",
    "years_since_last_sale"
]

TARGET = "price"

CATEGORICAL_FEATURES = [

    "status",
    "city",
    "state",
    "zip_code"
]

In [25]:
import os

# MinIO / S3
client = MlflowClient()
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://localhost:9000"

# Credenciales MinIO
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "supersecret"

# Opcional pero recomendado para MinIO
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DB = "mlops_db"
MYSQL_USER = "mlops_user"
MYSQL_PASSWORD = "mlops_pass"

MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENT = "diabetes_readmission"

MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "supersecret"

MINIO_BUCKET = "mlflow-artifacts"
BUCKET_LOGS = "pipeline-logs"

CLEAN_TABLE = "dataset_split"

TARGET_COLUMN = "target"

DROP_COLUMNS = [
    "readmitted",
    TARGET_COLUMN
]

MODEL_NAME = "real_estate_price_model"

MLFLOW_EXPERIMENT = (
    "real_estate_price_prediction"
)

mlflow.set_tracking_uri(
    MLFLOW_TRACKING_URI
)

mlflow.set_experiment(
    MLFLOW_EXPERIMENT
)

MYSQL_DATABASE = "mlops_db"

ENGINE = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)



In [4]:
from minio import Minio
from io import BytesIO
from datetime import datetime

PIPELINE_LOG_BUCKET = "pipeline-logs"

minio_client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

In [5]:
def ensure_log_bucket():

    if not minio_client.bucket_exists(
        PIPELINE_LOG_BUCKET
    ):
        minio_client.make_bucket(
            PIPELINE_LOG_BUCKET
        )

In [27]:
def ensure_minio_buckets():

    client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_ACCESS_KEY,
        secret_key=MINIO_SECRET_KEY,
        secure=False
    )

    for bucket in [
        MINIO_BUCKET,
        BUCKET_LOGS
    ]:

        if not client.bucket_exists(bucket):
            client.make_bucket(bucket)

    print("MinIO buckets verified")

In [6]:
def append_pipeline_log(text_message):

    ensure_log_bucket()

    object_name = "pipeline_history.txt"

    try:

        response = minio_client.get_object(
            PIPELINE_LOG_BUCKET,
            object_name
        )

        current_content = (
            response.read()
            .decode("utf-8")
        )

    except:

        current_content = ""

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    new_content = (
        current_content
        + "\n"
        + "=" * 80
        + "\n"
        + timestamp
        + "\n\n"
        + text_message
        + "\n"
    )

    data = new_content.encode("utf-8")

    minio_client.put_object(
        PIPELINE_LOG_BUCKET,
        object_name,
        BytesIO(data),
        len(data),
        content_type="text/plain"
    )

In [35]:
def decide_training():

    reasons = []

    drift_threshold = 0.10

    stats = pd.read_sql(
        """
        SELECT *
        FROM batch_statistics
        ORDER BY batch_number
        """,
        ENGINE
    )

    batches = sorted(
        stats["batch_number"].unique()
    )

    # Primer batch

    if len(batches) <= 1:

        reasons.append(
            "First batch detected. No trained model available."
        )

        return {
            "should_train": True,
            "decision": "TRAIN",
            "reasons": reasons
        }

    current_batch = batches[-1]
    previous_batch = batches[-2]

    current_stats = stats[
        stats["batch_number"] == current_batch
    ]

    previous_stats = stats[
        stats["batch_number"] == previous_batch
    ]

    train = False

    numeric_features = [
        "price",
        "bed",
        "bath",
        "acre_lot",
        "house_size"
    ]

    for feature in numeric_features:

        curr = current_stats[
            current_stats["feature_name"] == feature
        ]

        prev = previous_stats[
            previous_stats["feature_name"] == feature
        ]

        if curr.empty or prev.empty:
            continue

        curr_mean = curr.iloc[0]["mean_value"]
        prev_mean = prev.iloc[0]["mean_value"]

        curr_std = curr.iloc[0]["std_value"]
        prev_std = prev.iloc[0]["std_value"]

        if pd.notna(curr_mean) and pd.notna(prev_mean):

            pct_mean = abs(
                curr_mean - prev_mean
            ) / abs(prev_mean)

            if pct_mean > drift_threshold:

                train = True

                reasons.append(
                    f"{feature} mean drift "
                    f"{pct_mean:.2%} > "
                    f"{drift_threshold:.0%}"
                )

        if (
            pd.notna(curr_std)
            and pd.notna(prev_std)
            and prev_std > 0
        ):

            pct_std = abs(
                curr_std - prev_std
            ) / prev_std

            if pct_std > drift_threshold:

                train = True

                reasons.append(
                    f"{feature} std drift "
                    f"{pct_std:.2%} > "
                    f"{drift_threshold:.0%}"
                )

    categorical = current_stats[
        current_stats["new_categories_count"]
        .fillna(0)
        > 0
    ]

    for _, row in categorical.iterrows():

        train = True

        reasons.append(
            f"{row['feature_name']} introduced "
            f"{int(row['new_categories_count'])} "
            f"new categories"
        )

    if not train:

        reasons.append(
            "No numerical drift above threshold"
        )

        reasons.append(
            "No new categories detected"
        )

    return {
        "should_train": train,
        "decision": (
            "TRAIN"
            if train
            else "SKIP"
        ),
        "reasons": reasons
    }

In [36]:
def skip_training(decision):

    text = [
        "",
        "TRAINING DECISION",
        "",
        "Result:",
        "SKIP TRAINING",
        "",
        "Reasons:",
        ""
    ]

    for reason in decision["reasons"]:
        text.append(
            f"- {reason}"
        )

    append_pipeline_log(
        "\n".join(text)
    )

In [9]:
def load_training_dataset():

    query = """
    SELECT *
    FROM clean_data
    """

    df = pd.read_sql(
        query,
        ENGINE
    )

    return df

In [10]:
def build_training_metadata(df):

    metadata = {

        "numeric_ranges": {},

        "allowed_categories": {}
    }

    numeric_features = [

        "bed",
        "bath",
        "acre_lot",
        "house_size",
        "years_since_last_sale"
    ]

    for col in numeric_features:

        metadata["numeric_ranges"][col] = {

            "min": float(
                df[col].min()
            ),

            "max": float(
                df[col].max()
            )
        }

    for col in CATEGORICAL_FEATURES:

        metadata[
            "allowed_categories"
        ][col] = sorted(
            df[col]
            .astype(str)
            .dropna()
            .unique()
            .tolist()
        )

    return metadata

In [33]:
def train_evaluate_register_candidate():

    df = load_training_dataset()

    train_df = df[
        df["dataset_split"] == "train"
    ].copy()

    test_df = df[
        df["dataset_split"] == "test"
    ].copy()

    X_train = train_df[FEATURES]

    y_train = train_df[TARGET]

    X_test = test_df[FEATURES]

    y_test = test_df[TARGET]

    model = CatBoostRegressor(

        iterations=500,

        learning_rate=0.05,

        depth=6,

        loss_function="RMSE",

        verbose=False,

        random_seed=42
    )

    with mlflow.start_run() as run:

        model.fit(
            X_train,
            y_train,
            cat_features=CATEGORICAL_FEATURES
        )

        predictions = model.predict(
            X_test
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                predictions
            )
        )

        mae = mean_absolute_error(
            y_test,
            predictions
        )

        r2 = r2_score(
            y_test,
            predictions
        )

        mlflow.log_metric(
            "rmse",
            rmse
        )

        mlflow.log_metric(
            "mae",
            mae
        )

        mlflow.log_metric(
            "r2",
            r2
        )

        metadata = build_training_metadata(
            df
        )

        with tempfile.TemporaryDirectory() as tmpdir:

            metadata_path = os.path.join(
                tmpdir,
                "training_metadata.json"
            )

            with open(
                metadata_path,
                "w"
            ) as f:

                json.dump(
                    metadata,
                    f,
                    indent=4
                )

            mlflow.log_artifact(
                metadata_path
            )

        model_info = mlflow.catboost.log_model(
            model,
            name="model"
        )

        run_id = run.info.run_id

    return {

        "run_id": run_id,

        "model_uri":
            model_info.model_uri,

        "rmse": rmse,

        "mae": mae,

        "r2": r2
    }

In [12]:
def register_candidate_model(
    candidate
):

    registered_model = (
        mlflow.register_model(
            model_uri=candidate[
                "model_uri"
            ],
            name=MODEL_NAME
        )
    )

    return registered_model.version

In [42]:
def get_current_champion():

    try:

        champion = client.get_model_version_by_alias(
            MODEL_NAME,
            "champion"
        )

        run_id = champion.run_id

        run = client.get_run(run_id)

        rmse = run.data.metrics.get("rmse")

        return {
            "version": champion.version,
            "run_id": run_id,
            "rmse": rmse
        }

    except Exception:

        return None

In [14]:
def get_champion_rmse(
    champion_version
):

    version = (
        client.get_model_version(
            MODEL_NAME,
            champion_version
        )
    )

    run_id = version.run_id

    run = mlflow.get_run(
        run_id
    )

    return run.data.metrics[
        "rmse"
    ]

In [43]:
def compare_and_decide_promotion(
    candidate_metrics
):

    champion = get_current_champion()

    # Primer modelo

    if champion is None:

        return {
            "promote": True,
            "reason": (
                "No Champion model found. "
                "First model automatically promoted."
            ),
            "champion_rmse": None,
            "candidate_rmse": candidate_metrics["rmse"],
            "improvement_pct": None
        }

    champion_rmse = champion["rmse"]
    candidate_rmse = candidate_metrics["rmse"]

    # Protección por si champion no tiene rmse

    if champion_rmse is None:

        return {
            "promote": True,
            "reason": (
                "Champion model exists but "
                "RMSE metric was not found. "
                "Candidate promoted."
            ),
            "champion_rmse": None,
            "candidate_rmse": candidate_rmse,
            "improvement_pct": None
        }

    improvement = (
        (champion_rmse - candidate_rmse)
        / champion_rmse
    )

    promote = (
        candidate_rmse < champion_rmse
    )

    if promote:

        reason = (
            f"Promotion approved. "
            f"Candidate RMSE={candidate_rmse:.4f} "
            f"improves Champion RMSE={champion_rmse:.4f}. "
            f"Relative improvement={improvement:.2%}."
        )

    else:

        reason = (
            f"Promotion rejected. "
            f"Candidate RMSE={candidate_rmse:.4f} "
            f"is not better than Champion RMSE={champion_rmse:.4f}. "
            f"Relative improvement={improvement:.2%}."
        )

    return {
        "promote": promote,
        "reason": reason,
        "champion_rmse": champion_rmse,
        "candidate_rmse": candidate_rmse,
        "improvement_pct": improvement
    }

In [38]:
def promote_model(
    version,
    candidate_metrics,
    promotion
):

    client.set_registered_model_alias(
        MODEL_NAME,
        "champion",
        version
    )

    current_batch = pd.read_sql(
        """
        SELECT MAX(batch_number)
        AS batch_number
        FROM clean_data
        """,
        ENGINE
    ).iloc[0, 0]

    with ENGINE.begin() as conn:

        conn.execute(
            text(
                """
                UPDATE batch_statistics
                SET used_for_training = TRUE
                WHERE used_for_training = FALSE
                """
            )
        )

    training_run = pd.DataFrame([{

        "triggering_batch":
            current_batch,

        "mlflow_run_id":
            candidate_metrics["run_id"],

        "candidate_rmse":
            candidate_metrics["rmse"],

        "candidate_mae":
            candidate_metrics["mae"],

        "candidate_r2":
            candidate_metrics["r2"],

        "promoted":
            True,

        "created_at":
            pd.Timestamp.now()
    }])

    training_run.to_sql(

        "training_runs",

        ENGINE,

        if_exists="append",

        index=False
    )

    append_pipeline_log(
f"""
PROMOTION DECISION

Decision:
PROMOTE

Reason:
{promotion['reason']}

Champion RMSE:
{promotion['champion_rmse']}

Candidate RMSE:
{promotion['candidate_rmse']}

Improvement:
{promotion['improvement_pct']:.2%}

Version:
{version}
"""
    )

    print(
        f"Champion updated -> {version}"
    )

In [39]:
def reject_model(
    candidate_metrics,
    promotion
):

    current_batch = pd.read_sql(
        """
        SELECT MAX(batch_number)
        AS batch_number
        FROM clean_data
        """,
        ENGINE
    ).iloc[0, 0]

    training_run = pd.DataFrame([{

        "triggering_batch":
            current_batch,

        "mlflow_run_id":
            candidate_metrics["run_id"],

        "candidate_rmse":
            candidate_metrics["rmse"],

        "candidate_mae":
            candidate_metrics["mae"],

        "candidate_r2":
            candidate_metrics["r2"],

        "promoted":
            False,

        "created_at":
            pd.Timestamp.now()
    }])

    training_run.to_sql(

        "training_runs",

        ENGINE,

        if_exists="append",

        index=False
    )

    append_pipeline_log(
f"""
PROMOTION DECISION

Decision:
REJECT

Reason:
{promotion['reason']}

Champion RMSE:
{promotion['champion_rmse']}

Candidate RMSE:
{promotion['candidate_rmse']}

Improvement:
{promotion['improvement_pct']:.2%}
"""
)

    print(
        "Candidate rejected"
    )

In [18]:
def end_pipeline():

    append_pipeline_log(
        "Pipeline finished successfully"
    )

    print(
        "=" * 80
    )

    print(
        "PIPELINE FINISHED"
    )

    print(
        "=" * 80
    )

In [44]:
def simulate_training_dag():

    print(
        "\nSTART TRAINING DAG\n"
    )

    decision = decide_training()

    if not decision["should_train"]:
    
        skip_training(
            decision
        )
    
        end_pipeline()
    
        return
    
    append_pipeline_log(
        "\n".join(
            [
                "TRAINING DECISION",
                "",
                "Result:",
                "TRAIN",
                "",
                "Reasons:",
                *[
                    f"- {r}"
                    for r in decision["reasons"]
                ]
            ]
        )
    )
    
    candidate_metrics = (
        train_evaluate_register_candidate()
    )
    
    version = register_candidate_model(
        candidate_metrics
    )
    
    promotion = (
        compare_and_decide_promotion(
            candidate_metrics
        )
    )
    
    if promotion["promote"]:
    
        promote_model(
            version,
            candidate_metrics,
            promotion
        )
    
    else:
    
        reject_model(
            candidate_metrics,
            promotion
        )
    
    end_pipeline()

In [45]:
simulate_training_dag()


START TRAINING DAG



Registered model 'real_estate_price_model' already exists. Creating a new version of this model...
2026/06/01 21:57:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: real_estate_price_model, version 4


🏃 View run bustling-moose-853 at: http://localhost:5000/#/experiments/1/runs/579cef593f9b46bb9b0e3d8cddf46ec2
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '4' of model 'real_estate_price_model'.


Candidate rejected
PIPELINE FINISHED
